# Thermotec customer-enquiry POC

This notebook links Thermotec SKU rows from the product-master workbook to the validated Markdown family files. It produces a validation report and an internal callback-reference dataset. The local Streamlit demo may recommend a manufacturer-supported family, but this notebook does not select a SKU or change the source workbook or Google Sheet.

In [ ]:
from pathlib import Path
import re
import pandas as pd

def find_repo_root():
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / 'knowledge' / 'thermotec').is_dir():
            return candidate
    raise FileNotFoundError('Could not locate the repository root. Open this notebook from the cloned repository.')

REPO_ROOT = find_repo_root()
SOURCE_XLSX = REPO_ROOT / 'data' / 'raw' / 'Product_Master_Bot.xlsx'
SHEET_NAME = 'Sheet1'
MANUFACTURER = 'Thermotec'

print('Repository:', REPO_ROOT)
print('Source workbook:', SOURCE_XLSX)

## 1. Load the current product master

If the next cell reports that the file is missing, follow `POC_SETUP.md` and export the Google Sheet to the displayed location.

In [ ]:
if not SOURCE_XLSX.exists():
    raise FileNotFoundError(f'Export the Google Sheet as: {SOURCE_XLSX}')

products = pd.read_excel(SOURCE_XLSX, sheet_name=SHEET_NAME, dtype=object).fillna('')
required = {'Our SKU', 'SKU', 'Our Product Name', 'Manufacturer Name', 'Spec Family Name',
            'Thermal R Value', 'Acoustic Rw', 'NRC / αw', 'Validation Status',
            'Bot Content Status', 'External Validation'}
missing = sorted(required - set(products.columns))
if missing:
    raise ValueError(f'Missing required columns: {missing}')

thermotec = products[products['Manufacturer Name'].astype(str).str.strip().str.casefold() == MANUFACTURER.casefold()].copy()
thermotec.insert(0, 'Source Row', thermotec.index + 2)
print(f'Loaded {len(products):,} product rows; found {len(thermotec):,} Thermotec rows.')

## 2. Map each SKU to one product family

Rules are ordered from specific variants to broad families. This prevents foil-faced or UV-treated NuWave products inheriting the base NuWave record.

In [ ]:
FAMILY_RULES = [
    (r'4\s*-?\s*zero.*foil.*mlv|foil.*mlv.*nuwave', 'THERMOTEC_NUWAVE_FOIL_FACED_MLV'),
    (r'fence.*nuwave|uv\s*treated.*nuwave', 'THERMOTEC_NUWAVE_FENCE_MLV'),
    (r'underlay.*nuwave|nuwave.*underlay', 'THERMOTEC_NUWAVE_UNDERLAY'),
    (r'nuwrap\s*5', 'THERMOTEC_NUWRAP_5'),
    (r'nuwrap.*xtraflex|xtraflex', 'THERMOTEC_NUWRAP_XTRAFLEX'),
    (r'nuwave\s*base', 'THERMOTEC_NUWAVE_BASE_MLV'),
    (r'e[- ]?therm', 'THERMOTEC_E_THERM'),
    (r'e[- ]?flex\s*ht', 'THERMOTEC_E_FLEX_HT'),
    (r'e[- ]?flex\s*st', 'THERMOTEC_E_FLEX_ST'),
    (r'stonewool|rockwool|\bspi\b|rw\s*120kg', 'THERMOTEC_ROCKWOOL_PIPE'),
    (r'maxtape', 'THERMOTEC_MAXTAPE_FR'),
    (r'maxflex\s*coil', 'THERMOTEC_MAXFLEX_PIPE'),
    (r'4[- ]?zero', 'THERMOTEC_4_ZERO'),
]

def map_family(row):
    text = ' | '.join(str(row.get(c, '')) for c in ['Our SKU', 'SKU', 'Our Product Name', 'Spec Family Name']).casefold()
    for pattern, family_id in FAMILY_RULES:
        if re.search(pattern, text, flags=re.IGNORECASE):
            return family_id
    return ''

thermotec['Mapped Family ID'] = thermotec.apply(map_family, axis=1)
thermotec['Mapped Family ID'].replace('', '<UNMAPPED>').value_counts()

## 3. Load the Markdown knowledge index

In [ ]:
def read_frontmatter(path):
    text = path.read_text(encoding='utf-8')
    if not text.startswith('---\n'):
        return {}
    block = text.split('---', 2)[1]
    result = {}
    for line in block.splitlines():
        if ':' in line:
            key, value = line.split(':', 1)
            result[key.strip()] = value.strip()
    return result

knowledge = {}
for path in sorted((REPO_ROOT / 'knowledge' / 'thermotec').glob('*.md')):
    meta = read_frontmatter(path)
    family_id = meta.get('family_id')
    if family_id:
        knowledge[family_id] = {**meta, 'relative_path': path.relative_to(REPO_ROOT).as_posix()}

print(f'Loaded {len(knowledge)} product-family knowledge files.')
pd.DataFrame(knowledge).T[['canonical_name', 'validation_status', 'relative_path']].sort_index()

## 4. Apply the conservative enquiry-information gate

A row is usable as internal enquiry context only when it maps to an existing family file and its spreadsheet validation fields are ready/supported. This may support a family-level recommendation in local demo mode, but never authorises SKU, grade, quantity, compliance or installed-performance selection. Family documentation alone does not validate a SKU-specific value.

In [ ]:
thermotec['Knowledge File'] = thermotec['Mapped Family ID'].map(lambda x: knowledge.get(x, {}).get('relative_path', ''))
thermotec['Knowledge Status'] = thermotec['Knowledge File'].map(lambda x: 'LINKED' if x else 'MISSING KNOWLEDGE FILE')
profile_fields = {
    'Sustainability Score': 'priority_sustainability_score',
    'Energy Efficiency Score': 'priority_energy_efficiency_score',
    'Airborne Acoustic Comfort Score': 'priority_airborne_acoustic_comfort_score',
    'Compliance Gate': 'gate_ncc_project_compliance',
    'BAL Gate': 'gate_bal',
}
for output_column, metadata_key in profile_fields.items():
    thermotec[output_column] = thermotec['Mapped Family ID'].map(lambda x: knowledge.get(x, {}).get(metadata_key, ''))

source_ready = (
    thermotec['Bot Content Status'].astype(str).str.upper().eq('READY')
    & thermotec['External Validation'].astype(str).str.upper().isin({'SUPPORTED', 'VERIFIED'})
)
thermotec['Usable for Enquiry'] = thermotec['Knowledge Status'].eq('LINKED') & source_ready

def poc_note(row):
    if not row['Mapped Family ID']:
        return 'No family rule matched.'
    if row['Knowledge Status'] != 'LINKED':
        return 'Create and validate the missing family Markdown file.'
    if not row['Usable for Enquiry']:
        return 'Family is linked, but the spreadsheet source/status still needs validation.'
    return 'May support the callback brief and a family-level demo recommendation; SKU selection remains human-controlled.'

thermotec['POC Notes'] = thermotec.apply(poc_note, axis=1)

# Guard against the common stonewool density/Rw interpretation error.
density_name = thermotec['Our Product Name'].astype(str).str.contains(r'RW\s*120kg', case=False, regex=True)
bad_rw = ~thermotec['Acoustic Rw'].astype(str).str.strip().isin({'', 'Not stated', 'NA'})
thermotec.loc[density_name & bad_rw, 'POC Notes'] = 'CHECK: 120 kg/m³ is density, not an acoustic Rw rating.'
thermotec.loc[density_name & bad_rw, 'Usable for Enquiry'] = False

summary = thermotec.groupby(['Mapped Family ID', 'Knowledge Status'], dropna=False).agg(
    SKU_Count=('Our SKU', 'size'),
    Enquiry_Usable=('Usable for Enquiry', 'sum')
).reset_index().sort_values(['Knowledge Status', 'Mapped Family ID'])
summary

## 5. Export the audit report and internal callback reference

In [ ]:
report_dir = REPO_ROOT / 'reports'
output_dir = REPO_ROOT / 'data' / 'processed'
report_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)

report_columns = [
    'Source Row', 'Our SKU', 'SKU', 'Our Product Name', 'Active?', 'Category', 'Material Type',
    'Spec Family Name', 'Mapped Family ID', 'Knowledge File', 'Knowledge Status',
    'Thermal R Value', 'Acoustic Rw', 'NRC / αw', 'Validation Status',
    'Bot Content Status', 'External Validation', 'Sustainability Score',
    'Energy Efficiency Score', 'Airborne Acoustic Comfort Score', 'Compliance Gate',
    'BAL Gate', 'Usable for Enquiry', 'POC Notes'
]
report = thermotec[report_columns].copy()
report.to_csv(report_dir / 'thermotec_validation_report.csv', index=False, encoding='utf-8-sig')

bot_columns = [
    'Our SKU', 'SKU', 'Our Product Name', 'Category', 'Material Type', 'Product Use',
    'Mapped Family ID', 'Knowledge File', 'Thermal R Value', 'Acoustic Rw', 'NRC / αw',
    'Sustainability Score', 'Energy Efficiency Score', 'Airborne Acoustic Comfort Score',
    'Compliance Gate', 'BAL Gate',
    'Bot: Verified Features', 'Bot: Evidence-based Description',
    'Bot: Installation Guidance', 'Bot: Limitations / Warnings', 'Bot: Selection Summary'
]
callback_reference = thermotec.loc[thermotec['Usable for Enquiry'], bot_columns].copy()
callback_reference.to_csv(output_dir / 'thermotec_callback_reference.csv', index=False, encoding='utf-8-sig')

print('Thermotec rows:', len(thermotec))
print('Mapped rows:', thermotec['Mapped Family ID'].ne('').sum())
print('Rows with knowledge files:', thermotec['Knowledge File'].ne('').sum())
print('Rows usable as internal enquiry context:', int(thermotec['Usable for Enquiry'].sum()))
print('Demo recommendation scope: manufacturer-supported family only; no SKU, grade, quantity or compliance selection')
print('\nCreated:')
print(' -', report_dir / 'thermotec_validation_report.csv')
print(' -', output_dir / 'thermotec_callback_reference.csv')